# Detailed evaluation — reader ← GNN

Reads `evaluated_result_detailed.csv`, written by `src/evaluation/evaluate_detailed.py`.
Every metric is described in `docs/detailed_evaluation.md`. The last section compares this arm
paper by paper with another arm's detailed CSV, if one exists.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

ARM = 'reader ← GNN'
df = pd.read_csv('evaluated_result_detailed.csv')

NUMERIC = ['rouge1_f', 'rouge2_f', 'rougeL_f', 'bertscore_p', 'bertscore_r', 'bertscore_f1',
           'llm_coherence', 'llm_language',
           'field_match', 'contribution_match', 'modality_jaccard', 'method_family_match',
           'subfield', 'problem', 'task',
           'claims_total', 'claims_supported', 'claims_contradicted', 'claim_precision',
           'id_rank', 'id_correct', 'pair_score',
           'tfidf_rank', 'ontarget_gold_sim', 'ontarget_max_ref_sim', 'ontarget_margin']
NUMERIC = [c for c in NUMERIC if c in df.columns]   # pair_score only exists with --against
for col in NUMERIC:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df['tfidf_top1'] = (df['tfidf_rank'] == 1).astype(float)

# ── group num_of_neighbors into bins of 5 (1-5, 6-10, 11-15, …) ──────────────
BIN_SIZE = 5
max_neighbors = int(df['num_of_neighbors'].max())
bin_edges = list(range(0, max_neighbors + BIN_SIZE, BIN_SIZE))
NEIGHBOR_GROUPS = [f'{lo + 1}-{lo + BIN_SIZE}' for lo in bin_edges[:-1]]
df['neighbor_group'] = pd.cut(
    df['num_of_neighbors'], bins=bin_edges, labels=NEIGHBOR_GROUPS, right=True
)

def weighted_group_mean(frame: pd.DataFrame, value_col: str) -> pd.Series:
    """Mean of value_col per neighbour_group, weighted by num_of_neighbors."""
    def _wavg(g):
        mask = g[value_col].notna()
        if not mask.any():
            return np.nan
        return np.average(g.loc[mask, value_col], weights=g.loc[mask, 'num_of_neighbors'])
    return frame.groupby('neighbor_group', observed=True)[[value_col, 'num_of_neighbors']].apply(_wavg).reindex(NEIGHBOR_GROUPS)

print(f'{len(df)} rows')
df.head(3)

In [ ]:
# ── shared plot style ──────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.35,
    'grid.linestyle': '--',
})

def bar_plot(series: pd.Series, title: str, ylabel: str,
             color: str, ylim=None, int_yaxis=False):
    """Bar chart with one bar per neighbour-count group (e.g. 1-5, 6-10, …)."""
    values = [series.get(g, np.nan) for g in NEIGHBOR_GROUPS]
    x = np.arange(len(NEIGHBOR_GROUPS))

    fig, ax = plt.subplots(figsize=(11, 4))
    bars = ax.bar(x, values, color=color, edgecolor='white', linewidth=0.6)

    # label bars that have data
    for bar, v in zip(bars, values):
        if not np.isnan(v):
            label = f'{int(v)}' if int_yaxis else f'{v:.3f}'
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.004 * (bar.get_height() or 1),
                    label, ha='center', va='bottom', fontsize=7)

    ax.set_xticks(x)
    ax.set_xticklabels(NEIGHBOR_GROUPS, rotation=45, ha='right')
    ax.set_xlabel('Number of neighbours', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold', pad=10)
    if ylim:
        ax.set_ylim(*ylim)
    if int_yaxis:
        ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    fig.tight_layout()
    plt.show()

def dist_plot(col: str, values: list, title: str, xlabel: str, color: str):
    """Count of papers per score value."""
    counts = df[col].dropna().round().astype(int).value_counts().reindex(values, fill_value=0)
    fig, ax = plt.subplots(figsize=(8, 3.5))
    bars = ax.bar([str(v) for v in values], counts.values, color=color, edgecolor='white', linewidth=0.6)
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                str(int(v)), ha='center', va='bottom', fontsize=9)
    ax.set_title(title, fontsize=13, fontweight='bold', pad=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Number of abstracts')
    ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    fig.tight_layout()
    plt.show()
    return counts

## Summary: every metric

In [ ]:
summary = pd.DataFrame({
    'mean': df[NUMERIC + ['tfidf_top1']].mean(),
    'median': df[NUMERIC + ['tfidf_top1']].median(),
    'n': df[NUMERIC + ['tfidf_top1']].notna().sum(),
}).round(3)
summary

## Kept from the current evaluator

In [ ]:
# ── ROUGE-L and BERTScore by neighbours ──────────────────────────────────────
rougel_mean = weighted_group_mean(df, 'rougeL_f')
bar_plot(rougel_mean, title=f'Mean ROUGE-L F1 by Number of Neighbours — {ARM}',
         ylabel='Mean ROUGE-L F1', color='#4C72B0', ylim=(0, max(rougel_mean.dropna()) * 1.2))

bert_mean = weighted_group_mean(df, 'bertscore_f1')
bar_plot(bert_mean, title=f'Mean BERTScore F1 by Number of Neighbours — {ARM}',
         ylabel='Mean BERTScore F1', color='#55A868',
         ylim=(bert_mean.dropna().min() * 0.98, bert_mean.dropna().max() * 1.01))

In [ ]:
# ── Coherence and language ───────────────────────────────────────────────────
bar_plot(weighted_group_mean(df, 'llm_coherence'),
         title=f'Mean LLM Coherence (1–10) by Number of Neighbours — {ARM}',
         ylabel='Mean LLM Coherence', color='#DD8452', ylim=(0, 10))
bar_plot(weighted_group_mean(df, 'llm_language'),
         title=f'Mean LLM Language (1–10) by Number of Neighbours — {ARM}',
         ylabel='Mean LLM Language', color='#8172B2', ylim=(0, 10))
dist_plot('llm_coherence', list(range(1, 11)), f'Distribution of LLM Coherence — {ARM}',
          'LLM Coherence score (1–10)', '#DD8452')

## A. Blind facets

Each abstract is labelled on its own from fixed lists; the score is whether the generated
label matches the gold label. Field is expected near the ceiling for every arm.

In [ ]:
FACETS = ['field_match', 'contribution_match', 'modality_jaccard', 'method_family_match']
FACET_LABELS = ['Field', 'Contribution type', 'Modality (Jaccard)', 'Method family']
rates = df[FACETS].mean().values

fig, ax = plt.subplots(figsize=(8, 3.5))
bars = ax.bar(FACET_LABELS, rates, color='#4C72B0', edgecolor='white', linewidth=0.6)
for bar, v in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.01, f'{v:.1%}', ha='center', va='bottom', fontsize=9)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Match rate')
ax.set_title(f'Blind facet agreement with the gold — {ARM}', fontsize=13, fontweight='bold', pad=10)
fig.tight_layout()
plt.show()

for col in FACETS:
    bar_plot(weighted_group_mean(df, col), title=f'{col} by Number of Neighbours — {ARM}',
             ylabel=col, color='#4C72B0', ylim=(0, 1.05))

In [ ]:
# ── Confusion tables: gold label (rows) vs generated label (columns) ─────────
def confusion(facet: str):
    table = pd.crosstab(df[f'gold_{facet}'], df[f'gen_{facet}'])
    labels = sorted(set(table.index) | set(table.columns))
    table = table.reindex(index=labels, columns=labels, fill_value=0)
    fig, ax = plt.subplots(figsize=(0.55 * len(labels) + 3, 0.45 * len(labels) + 2))
    ax.imshow(table.values, cmap='Blues')
    ax.grid(False)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=8)
    for i in range(len(labels)):
        for j in range(len(labels)):
            if table.values[i, j]:
                ax.text(j, i, table.values[i, j], ha='center', va='center', fontsize=7)
    ax.set_xlabel('generated')
    ax.set_ylabel('gold')
    ax.set_title(f'{facet}: gold vs generated — {ARM}', fontsize=12, fontweight='bold')
    fig.tight_layout()
    plt.show()

for facet in ['field', 'contribution', 'method_family']:
    confusion(facet)

## B. Comparison with the gold

Subfield, problem and task on the anchored 0–3 scale (0 different, 1 related, 2 same with
differences in detail, 3 same).

In [ ]:
SCALE = ['subfield', 'problem', 'task']
counts = pd.DataFrame({c: df[c].dropna().round().astype(int).value_counts().reindex(range(4), fill_value=0)
                       for c in SCALE})

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(4)
for k, (c, color) in enumerate(zip(SCALE, ['#4C72B0', '#55A868', '#C44E52'])):
    ax.bar(x + (k - 1) * 0.27, counts[c].values, width=0.27, label=f'{c} (mean {df[c].mean():.2f})',
           color=color, edgecolor='white', linewidth=0.6)
ax.set_xticks(x)
ax.set_xticklabels(['0 different', '1 related', '2 same, detail differs', '3 same'])
ax.set_ylabel('Number of abstracts')
ax.set_title(f'Subfield / problem / task — {ARM}', fontsize=13, fontweight='bold', pad=10)
ax.legend()
fig.tight_layout()
plt.show()

for c in SCALE:
    bar_plot(weighted_group_mean(df, c), title=f'Mean {c} (0–3) by Number of Neighbours — {ARM}',
             ylabel=f'Mean {c}', color='#C44E52', ylim=(0, 3))
counts

In [ ]:
# ── Claims: specificity, precision, contradictions ───────────────────────────
print(f"claims per abstract:        {df['claims_total'].mean():.2f}")
print(f"supported per abstract:     {df['claims_supported'].mean():.2f}")
print(f"contradicted per abstract:  {df['claims_contradicted'].mean():.2f}")
print(f"claim precision (mean):     {df['claim_precision'].mean():.3f}")
print(f"pooled precision:           {df['claims_supported'].sum() / df['claims_total'].sum():.3f}")
print(f"abstracts with ≥1 contradiction: {(df['claims_contradicted'] > 0).mean():.1%}")
print(f"abstracts with no claims:   {(df['claims_total'] == 0).mean():.1%}")

max_claims = int(df['claims_total'].max())
dist_plot('claims_total', list(range(0, max_claims + 1)), f'Claims per abstract — {ARM}',
          'claims_total', '#8172B2')

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(df['claim_precision'].dropna(), bins=np.linspace(0, 1, 11), color='#55A868', edgecolor='white')
ax.set_xlabel('claim_precision (supported / total)')
ax.set_ylabel('Number of abstracts')
ax.set_title(f'Claim precision — {ARM}', fontsize=13, fontweight='bold', pad=10)
fig.tight_layout()
plt.show()

## C. Identification among hard distractors

The judge ranks the gold and its four nearest papers (never its own references). Chance is
20% correct and a mean rank of 3.

In [ ]:
print(f"id correct (top-1): {df['id_correct'].mean():.1%}   (chance 20%)")
print(f"mean id rank:       {df['id_rank'].mean():.2f}   (chance 3.00)")
print(f"id MRR:             {(1 / df['id_rank']).mean():.3f}   (chance 0.457)")

counts = dist_plot('id_rank', [1, 2, 3, 4, 5], f'Rank of the true paper among 5 — {ARM}',
                   'rank (1 = identified)', '#4C72B0')
bar_plot(weighted_group_mean(df, 'id_correct'),
         title=f'Identification top-1 by Number of Neighbours — {ARM}',
         ylabel='Share identified', color='#4C72B0', ylim=(0, 1.05))

## No judge: TF-IDF identifiability and on-target margin

In [ ]:
print(f"TF-IDF top-1 (among all {len(df)} golds): {df['tfidf_top1'].mean():.1%}")
print(f"TF-IDF MRR:                         {(1 / df['tfidf_rank']).mean():.3f}")
print(f"on-target margin (mean):            {df['ontarget_margin'].mean():.3f}")
print(f"closer to a reference than to gold: {(df['ontarget_margin'] < 0).mean():.1%}")

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(df['ontarget_margin'].dropna(), bins=40, color='#CCB974', edgecolor='white')
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('similarity to gold − highest similarity to a reference')
ax.set_ylabel('Number of abstracts')
ax.set_title(f'On-target margin — {ARM}', fontsize=13, fontweight='bold', pad=10)
fig.tight_layout()
plt.show()

bar_plot(weighted_group_mean(df, 'ontarget_margin'),
         title=f'Mean on-target margin by Number of Neighbours — {ARM}',
         ylabel='Mean margin', color='#CCB974')

## D. Pairwise preference

Only present when the CSV was scored with `--against`. 1 = this arm judged better in both
orders, 0.5 = split or tie, 0 = worse in both.

In [ ]:
if 'pair_score' in df.columns:
    p = df['pair_score'].dropna()
    se = p.std() / np.sqrt(len(p))
    print(f"pair score: {p.mean():.3f} ± {se:.3f} (SE)   0.5 = no preference")
    print(f"wins both orders: {(p == 1).mean():.1%}   split/tie: {(p == 0.5).mean():.1%}   "
          f"loses both: {(p == 0).mean():.1%}")
    counts = p.value_counts().reindex([0.0, 0.25, 0.5, 0.75, 1.0], fill_value=0)
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.bar([str(v) for v in counts.index], counts.values, color='#937860', edgecolor='white')
    ax.set_xlabel('pair_score')
    ax.set_ylabel('Number of abstracts')
    ax.set_title(f'Pairwise preference — {ARM}', fontsize=13, fontweight='bold', pad=10)
    fig.tight_layout()
    plt.show()
    bar_plot(weighted_group_mean(df, 'pair_score'),
             title=f'Pair score by Number of Neighbours — {ARM}',
             ylabel='Mean pair score', color='#937860', ylim=(0, 1.05))
else:
    print('no pair_score column: this CSV was scored without --against')

## Examples

In [ ]:
pd.set_option('display.max_colwidth', 2000)
SHOW = ['node_id', 'original_abstract', 'generated_abstract']

# 2 examples per task score
(df[df['task'].notna()].sort_values('task').groupby('task', sort=True)
   .apply(lambda g: g.head(2)).reset_index(drop=True)[SHOW + ['subfield', 'problem', 'task']])

In [ ]:
# Identified by TF-IDF but not by the judge, and the reverse
judge_only = df[(df['id_correct'] == 1) & (df['tfidf_rank'] > 1)]
tfidf_only = df[(df['id_correct'] == 0) & (df['tfidf_rank'] == 1)]
print(f'judge only: {len(judge_only)}   TF-IDF only: {len(tfidf_only)}')
tfidf_only.head(3)[SHOW + ['id_rank', 'id_options']]

In [ ]:
# Most contradicted abstracts
df.sort_values('claims_contradicted', ascending=False).head(3)[
    SHOW + ['claims_total', 'claims_supported', 'claims_contradicted']]

## Paired comparison with another arm

Set `OTHER` to another arm's `evaluated_result_detailed.csv`. Differences are this arm minus
the other, per paper on the shared papers; SE = sd / √n. Positive means this arm is higher, which
is better for every metric except `claims_contradicted`, `id_rank` and `tfidf_rank`.

In [ ]:
OTHER = '../2026-10-09-14541449_run11-reader-refs-mean/evaluated_result_detailed.csv'
OTHER_NAME = 'reader ← refs-mean'

if os.path.exists(OTHER):
    other = pd.read_csv(OTHER)
    COMPARE = [c for c in NUMERIC + ['tfidf_top1'] if c != 'pair_score']
    for col in COMPARE:
        if col in other.columns:
            other[col] = pd.to_numeric(other[col], errors='coerce')
    other['tfidf_top1'] = (other['tfidf_rank'] == 1).astype(float)
    both = df.merge(other, on='node_id', suffixes=('', '_other'))
    rows = []
    for col in COMPARE:
        d = (both[col] - both[f'{col}_other']).dropna()
        rows.append({'metric': col, ARM: both[col].mean(), OTHER_NAME: both[f'{col}_other'].mean(),
                     'diff': d.mean(), 'SE': d.std() / np.sqrt(len(d)), 'n': len(d)})
    paired = pd.DataFrame(rows).set_index('metric')
    paired['z'] = paired['diff'] / paired['SE']
    display(paired.round(3))

    # Relative difference on the main per-dimension metrics
    MAIN = ['field_match', 'contribution_match', 'modality_jaccard', 'method_family_match',
            'subfield', 'problem', 'task', 'claim_precision', 'id_correct', 'tfidf_top1']
    rel = paired.loc[MAIN, 'diff'] / paired.loc[MAIN, OTHER_NAME]
    err = paired.loc[MAIN, 'SE'] / paired.loc[MAIN, OTHER_NAME]
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(MAIN, rel.values, yerr=1.96 * err.values, capsize=3,
           color=['#55A868' if v > 0 else '#C44E52' for v in rel.values], edgecolor='white')
    ax.axhline(0, color='black', linewidth=1)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    ax.set_ylabel(f'relative difference vs {OTHER_NAME}')
    ax.set_title(f'{ARM} vs {OTHER_NAME} (95% CI)', fontsize=13, fontweight='bold', pad=10)
    plt.xticks(rotation=30, ha='right')
    fig.tight_layout()
    plt.show()
else:
    print(f'{OTHER} not found yet')